In [ ]:
using Pkg
Pkg.add(["HTTP", "JSON3"])

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`


In [46]:
using HTTP, JSON3

function get_hello_message(name::String)
    return "Hello $name"
end

println(get_hello_message())

Hello World from ur mum gae


In [47]:
println("Hello, can you see this?")

Hello, can you see this?


In [52]:
using HTTP
using JSON3

function get_hello_message(name::String)
    return "Hi $name, the server is working!"
end

function handler(req)
    headers = [
        "Access-Control-Allow-Origin" => "*",
        "Access-Control-Allow-Methods" => "GET, POST, OPTIONS",
        "Access-Control-Allow-Headers" => "Content-Type",
        "Content-Type" => "application/json"
    ]

    println("Request target: ", req.target)

    if req.method == "OPTIONS"
        return HTTP.Response(200, headers, "")

    elseif req.method == "GET" && req.target == "/test"
        return HTTP.Response(200, headers, JSON3.write(Dict("message" => "Server is working!")))

    elseif req.method == "POST" && (req.target == "/hello" || req.target == "/hello/")
        raw_body = String(req.body)
        println("Raw body: ", raw_body)

        # Parse JSON
        data = JSON3.read(raw_body)

        if haskey(data, "name")
            name = String(data["name"])
            response = Dict("message" => get_hello_message(name))
            return HTTP.Response(200, headers, JSON3.write(response))
        else
            return HTTP.Response(400, headers, JSON3.write(Dict("error" => "Missing 'name' field")))
        end
    end

    return HTTP.Response(404, headers, JSON3.write(Dict("error" => "Not found")))
end

server_task = @async HTTP.serve!(handler, "0.0.0.0", 9090)

println("✅ Server running at http://localhost:9090/hello")


✅ Server running at http://localhost:9090/hello


In [53]:
schedule(server_task, InterruptException())
println("🛑 Server stopped")

ErrorException: schedule: Task not runnable